## Demo 2: StackExchange

The data of a StackExchange site, published as one XML file per table on archive.org.

This demo is about getting files into databases, and about moving data from one database into another one. We start by looking at the data, without touching a database at all.

### What data do we have?

In [ ]:
import os

data_path = r"..\data\stackexchange"

for file in sorted(os.listdir(data_path)):
    size = os.path.getsize(os.path.join(data_path, file))
    print(f"{file:20} {size / 1024 / 1024:6.1f} MB")

### The files are XML, but they are also line oriented

That is what makes them pleasant to work with: the whole file is valid XML, and every single row is valid XML on its own. So we never have to load the whole document into memory.

In [ ]:
users_path = r"..\data\stackexchange\Users.xml"

with open(users_path, encoding="utf-8-sig") as file:
    users_lines = file.readlines()

print(len(users_lines), "lines")
print(users_lines[0], end="")
print(users_lines[1], end="")
print(users_lines[2][:110], "...")
print(users_lines[-1])

#### Why `utf-8-sig` and not `utf-8`?

The file starts with a byte order mark. `Get-Content` in PowerShell removes it without telling us, `open` in Python does not. With plain `utf-8` the first line starts with an invisible `\ufeff`, and a test like `line.startswith("<?xml")` is suddenly false.

In [ ]:
for encoding in ["utf-8", "utf-8-sig"]:
    with open(users_path, encoding=encoding) as file:
        first_line = file.readline()

    print(f"{encoding:10} {first_line[:24]!r:36} starts with '<?xml': {first_line.startswith('<?xml')}")

### One line is one row

In PowerShell we cast the line to `[xml]` and get an `XmlElement` with one property per attribute. In Python we parse the line and take its attributes, and what we get is a plain dictionary.

In [ ]:
import xml.etree.ElementTree as ET

line = users_lines[2]

row = ET.fromstring(line).attrib

row

In [ ]:
print(type(row))
print(type(row["Id"]), repr(row["Id"]))

Every value is a string, in both languages. Nobody has converted anything yet - the types will come from the target table.

### Not every row has every attribute

And this is where the two languages really differ. An attribute that is not in the line is simply not in the dictionary.

In [ ]:
from collections import Counter

attribute_counts = Counter()
row_count = 0

for line in users_lines:
    if line.lstrip().startswith("<row"):
        row_count += 1
        attribute_counts.update(ET.fromstring(line).attrib.keys())

for attribute, count in attribute_counts.most_common():
    print(f"{attribute:16} {count:6} of {row_count}")

In [ ]:
# Find the first user without a Location

for line in users_lines:
    if line.lstrip().startswith("<row"):
        sparse_row = ET.fromstring(line).attrib
        if "Location" not in sparse_row:
            break

print(sparse_row["DisplayName"], "has no Location")

# PowerShell gives us $null for a missing property, and so does .get()
print(sparse_row.get("Location"))

In [ ]:
# But asking for it directly is an error, not a None

sparse_row["Location"]

So a Python port of the import cannot simply read `row[column]` for every column of the target table. It either asks with `.get()`, or it has to know which attributes are there.

### From lines to a data frame

pandas has its own answer to the missing attributes: it collects every key it sees and fills the gaps with `NaN`.

In [ ]:
import pandas as pd

users = pd.DataFrame(
    ET.fromstring(line).attrib
    for line in users_lines
    if line.lstrip().startswith("<row")
)

users

In [ ]:
users.info()

Twelve columns, all of them strings, and three of them with missing values. Converting those strings into the types of a database table is the next step.

### Setting up the connection to SQL Server

Same three lines as in the first demo, only the database and the login are different.

In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path("../lib").resolve()))

from connect_sql_instance import connect_sql_instance
from import_sql_table import import_sql_table
from invoke_sql_query import invoke_sql_query

connection = connect_sql_instance(
    instance="127.0.0.1",
    database="StackExchange",
    username="StackExchange",
    password="Passw0rd!"
)

### What does the target table look like?

The file gives us strings. The table decides what they have to become. `cursor.description` tells us the columns and, for each one, the Python type that pyodbc expects. It is what `GetSchemaTable()` is for the PowerShell version.

In [ ]:
cursor = connection.cursor()
cursor.execute("SELECT TOP 0 * FROM dbo.Users")
description = cursor.description
cursor.close()

for name, type_code, _, _, _, _, null_ok in description:
    print(f"{name:16} {type_code.__name__:10} null_ok={null_ok}")

#### The file and the table do not agree

The table has fourteen columns, a row in the file has twelve attributes, and not even the same twelve on every row.

In [ ]:
table_columns = [column[0] for column in description]

print("in the table but never in the file:", [c for c in table_columns if c not in attribute_counts])
print("in the file but not in the table:  ", [a for a in attribute_counts if a not in table_columns])

### Importing the file

`import_sql_table` reads the file line by line, so the size of the file does not matter. For every line it builds one value per column of the *target* table: it asks the row with `.get()`, so a missing attribute becomes `NULL`, converts the string with the type from `cursor.description`, and sends the rows to the database in batches.

In [ ]:
import_sql_table(
    connection=connection,
    path=r"..\data\stackexchange\Users.xml",
    table="dbo.Users",
    batch_size=5000,
    truncate_table=True
)

In [ ]:
invoke_sql_query(
    connection=connection,
    query='SELECT TOP 5 Id, DisplayName, Location, CreationDate, Reputation FROM dbo.Users'
)

The dates are dates and the numbers are numbers, and the two columns that the file never mentions are `NULL` for every row - just like the `Location` of the users that did not fill it in.

In [ ]:
invoke_sql_query(connection=connection, query="""
SELECT COUNT(*) AS ImportedRows,
       SUM(CASE WHEN Location IS NULL THEN 1 ELSE 0 END) AS NullLocation,
       SUM(CASE WHEN Age IS NULL THEN 1 ELSE 0 END) AS NullAge
FROM dbo.Users""")

### When the file names a column differently

Badges are created on `Date`, but every table in this database calls that column `CreationDate`.

In [ ]:
badges_path = r"..\data\stackexchange\Badges.xml"

with open(badges_path, encoding="utf-8-sig") as file:
    badges_lines = file.readlines()

ET.fromstring(badges_lines[2]).attrib

In [ ]:
import_sql_table(
    connection=connection,
    path=badges_path,
    table="dbo.Badges",
    batch_size=5000,
    truncate_table=True,
    column_map={"CreationDate": "Date"}
)

In [ ]:
invoke_sql_query(connection=connection, query='SELECT TOP 5 * FROM dbo.Badges')

### The one thing that is really different

The PowerShell version fills a `DataTable` whose columns are typed from `GetSchemaTable()`, and lets ADO.NET convert the strings on the way in. pyodbc has nothing like that: in fast bulk mode it binds a value by its Python type, so a string never reaches an `INT` column.

So `import_sql_table` carries a small table of converters - `int`, `str`, `datetime.fromisoformat` - and picks one per column from `cursor.description`. That table is the part of this function with no counterpart in the sibling repository.